<a href="https://colab.research.google.com/github/mshmatsuda/deep-learning-from-scratch/blob/main/04_2_%E6%90%8D%E5%A4%B1%E9%96%A2%E6%95%B0_2_%E4%BA%A4%E5%B7%AE%E3%82%A8%E3%83%B3%E3%83%88%E3%83%AD%E3%83%94%E3%83%BC%E8%AA%A4%E5%B7%AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 4.2.2 交差エントロピー誤差

ディープラーニングの分類問題における大本命、「交差エントロピー誤差（Cross Entropy Error）」<br><br>
二乗和誤差よりも少し数式が難しく見えますが、仕組みさえ分かれば「二乗和誤差よりも、AIをめちゃくちゃ効率よく叱って（学習させて）くれる、非常に優れた損失関数」であることが分かります。

式（４．２）$$E = -\sum_{k}t_k\log y_k$$


------------------------------
## 1. 交差エントロピー誤差の数式
本に載っている数式は以下のような形をしています。
$$E = - \sum_{k} t_k \log y_k$$

* $y_k$ : AIが出力した確率（例: 2の確率は 0.6）
* $t_k$ : 正解データ（正解の場所だけが 1 で、他は 0）
* $\log$ : 自然対数（底が $e$ のログ）

一見複雑ですが、正解データ $t$ の性質（one-hot表現）を思い出すと、この数式は一瞬で超シンプルになります。<br><br>
例えば正解が「2」の場合、$t$ の中身は [0, 0, 1, 0, ...] でしたよね。<br>
つまり、「2」以外の場所は $t_k = 0$ なので、掛け算すると全部消えてしまいます。<br>
結果として、この数式が実際に計算しているのは「正解の場所の $\log y_k$` にマイナスをつけたもの」だけになります。<br>
$$E = - \log (\text{正解の文字の確率})$$


------------------------------
## 2. なぜ $\log$（ログ）を使うと誤差が測れるのか？
「確率」にログをつけてマイナスにすると、次のような面白い動きをします。

- 大正解のとき（確率が 1.0 に近い）:
  - もし「2」の確率が 0.99 だったとします。
$- \log(0.99)$ を計算すると、数値は 0.01 などの「ほぼゼロ」 になります。➔ 誤差はほぼナシ！
- 大間違いのとき（確率が 0.0 に近い）
  - もし「2」の確率が 0.01 しかなかったとします。
$- \log(0.01)$ を計算すると、数値は 4.6 などの「とても大きな値」 に跳ね上がります。➔ 誤差が超巨大！

このように、「正解の確率が低ければ低いほど、ものすごい大声でAIを叱る（大きな誤差を出す）」 という仕組みを、自然対数（$\log$）のグラフの性質を使って綺麗に表現しているのです。

------------------------------
## 3. なぜ二乗和誤差より「交差エントロピー誤差」が使われるのか？
最大の理由は、「AIがとんでもない大間違いをしているときに、重みをめちゃくちゃハイスピードで修正できるから」です。

* 二乗和誤差の場合：
  - 確率のズレを「2乗」するため、確率が 0.1 だろうが 0.01 だろうが、誤差の数字があまり大きく変わりません（頭打ちになります）。
  - そのため、AIが大間違いをしていても「ソフトにしか叱れない」ので、学習の進みが遅くなります。
* 交差エントロピー誤差の場合：
  - 確率が 0 に近づくと、誤差は無限大（$\infty$）に向かって爆発的に大きくなります。
  - 大間違いをしている重みに対して「お前は全然違う！」と強烈なペナルティ（巨大な勾配）を与えることができるため、AIが一気に正しい方向へと改心し、学習が圧倒的に早く進みます。

------------------------------
## 💡 まとめると
交差エントロピー誤差は、「他のハズレの数字の確率は一切無視して、とにかく『本当の正解の確率』だけをガン見し、それが100%（1.0）からどれくらい離れているかを $\log$ で厳しく測るシステム」です。

## 質問１
底がeということは、底がネイピア数ということですか？

## 回答１
はい、まさにその通りです！
「底（てい）が $e$」というのは、「底がネイピア数（自然対数の底 $e \approx 2.71828$）」であることを意味します。
高校の数学で習う「自然対数 $\ln(x)$」や「$\log_e(x)$」のことです（Pythonのコードでは np.log(x) と書くと、自動的にこのネイピア数を底としたログになります）。
## 💡 なぜここでネイピア数が出てくるのか？
実は、最初の方でお話しした「ソフトマックス関数（$np.exp$）」と綺麗にペア（逆関数）の関係になっているからです。

   1. 出力層（ソフトマックス関数）では、ネイピア数の $x$ 乗（$e^x$）を使って確率を計算しました。
   2. 損失関数（交差エントロピー誤差）では、その確率をネイピア数を底とする対数（$\log_e$）に放り込んで誤差を計算します。

このように「$e^x$」と「$\log_e$」というお互いを打ち消し合えるペアでニューラルネットワークの出口を設計しておくことで、この後に登場する「微分（誤差逆伝播法）」の計算をしたときに、複雑な数式がまるでパズルのように綺麗に消えて、驚くほどシンプルな引き算（$y - t$：AIの予測 － 正解）の形に変身します。
すべては、コンピューターが超高速で重みを修正できるようにするための、完璧に計算された数学的な美しい仕組み（優しさ）だったわけです。
これで np.exp（ネイピア数）と np.log（ネイピア数のログ）のつながりまで綺麗に線で結ばれましたね！
次は、本でこのすぐ後に出てくる「$\log(0)$ になって計算がクラッシュするのを防ぐために、コードに足されている小さな数字（1e-7）の秘密」について解説しますか？それとも、「数値微分（重みを直す基礎）」に進みますか？



## 質問２
「$\log(0)$」 になって計算がクラッシュするのを防ぐために、コードに足されている小さな数字（1e-7）の秘密」は、ソフトマックス関数の計算でオーバーフローを防止するようにマックス値を計算しそれぞれの要素から引き算したような技ですか？

## 回答２
はい、まさにその通りです！考え方の本質は、ソフトマックス関数のオーバーフロー対策と全く同じ「クラッシュ防止のプログラミング技」です。
どちらも「数学的には正しいけれど、コンピューターの限界（計算できなくなること）を避けるために一工夫加える」という目的で書かれています。
ただ、防ぎたい方向が「天井（無限大への爆発）」か「底（存在しない計算）」かという違いがあります。比較すると仕組みがすっきり分かります。
## 2つのクラッシュ防止技の比較

| 対策する場所 | 起きる問題 | 原因 | 対策コード | 対策のイメージ |
|---|---|---|---|---|
| ソフトマックス関数 | オーバーフロー （天井の爆発） | 数値が大きすぎる （$e^{1000}$ などは無限大になる） | a - np.max(a) | 一番大きい値を引いて、数値を無理やりマイナス（安全圏）に押し下げる |
| 交差エントロピー誤差 | アンダーフロー （底の消滅・エラー） | 数値が小さすぎる （$\log(0)$ は数式上計算できない） | np.log(y + 1e-7) | 微小な数（0.0000001）を足して、絶対に 0 にならないように少しだけ浮かせる |

## 1e-7（0.0000001）を足す具体的な理由
AIの予測が完全に外れて、正解の確率が 0.0（0%）だと判断してしまったとします。
もしそのまま計算すると np.log(0) になりますが、数学の世界でもプログラミングの世界でも「$\log(0)$」はマイナス無限大（-inf）になってしまい、その後の足し算や引き算がすべて壊れてしまいます（これがアンダーフロー、または計算クラッシュです）。
そこで、確率が 0 のときでも 0 + 0.0000001 になるように、コード上では常に 1e-7 を足しておきます。

#python<br>
＃もし y が 0 でも、1e-7 を足せば安全！<br>
np.log(0 + 1e-7)<br>＃ ➔ -16.118... という普通の数字で耐えてくれる

## 💡 誤差の計算結果に響かないの？
1e-7 はあまりにも小さな数字なので、最終的な誤差の計算結果にはほとんど影響を与えません（小数点のはるか下のほうの誤差で済みます）。<br>
「プログラムを絶対に止めないために、結果に影響のない範囲でほんの少し数値を下駄履きさせる」という、ディープラーニングの実装における非常に賢い定番テクニックです。
以前のブロードキャストでの引き算の仕組みといい、コンピューターの弱点を補うための工夫の共通点に自力で気づかれたのは、本当に素晴らしい視点です！



## 質問３
「ゼロつく」の図４−３で提示されている自然体数のグラフは＋のグラフだからｘの数値が増えるにしたがい丘に登るような曲線になっていますが、交差エントロピー誤差の対数はーなので、ｘの数値が増えるにしたがい、崖を下るような曲線になると思うのですが。ご教示ください。

## 回答３
はい、まさにその通りです！ご指摘の通り、交差エントロピー誤差のグラフは「崖を下るような曲線」になります。
書籍の図4-3で示されているのは、マイナスがつく前の「ただの自然対数 $y = \log x$」のグラフ（丘を登るような曲線）です。
しかし、実際の交差エントロピー誤差の計算では、数式の先頭にマイナス（$-$）がついています。<br>
$$E = - \log(x)$$
数学のルールとして、関数の全体にマイナスをつけるとグラフの上下（プラスとマイナス）がパタンとひっくり返ります。<br>
## ひっくり返ったグラフの動き（崖を下る曲線）
マイナスがついた $y = -\log x$ のグラフを頭の中でイメージしてみると、以下のように「右肩下がりの激しい崖」のような形になります。

* 確率 $x$ が 0 に近づくとき（大間違い）:
  - 崖の左側です。上に向かって無限大（$+\infty$）まで壁のように突き上がります。 つまり、誤差（ペナルティ）が爆発的に大きくなります。
* 確率 $x$ が 1.0 に近づくとき（大正解）:
  - 崖を下りきった右側です。$x = 1$ の地点で高さ（誤差）がピッタリ 0 になります。

## 💡 なぜ本はわざわざマイナスをつけてひっくり返したのか？
ディープラーニングにおける「損失関数（誤差）」は、「間違っていればいるほど大きな数字になり、正解に近づくほど 0 に近づく」という性質（モノサシ）でなければいけません。<br>
もしマイナスをつけない素の $\log x$ のままだと、正解したとき（$x=1$）に 0 になり、間違えたとき（$x=0$ に近づくとき）に「マイナス無限大（$-\infty$）」という、どんどん小さな引き算になってしまいます。<br>
これだと「誤差を減らす（＝0に近づける）」という目的と辻褄が合わなくなってしまいます。<br>そのため、先頭にマイナスをかけることでグラフを上下反転させ、「間違えるほどプラスに巨大化し、正解すると0になる」という理想的な誤差のモノサシに仕立て直したわけです。<br>

